# Emotion Detection from Scratch

**Author:** Suchit Mathur  
**LinkedIn:** https://www.linkedin.com/in/mathursuchit/  
**Dataset:** FER-2013 (Kaggle) — 35,000+ grayscale face images, 7 emotion classes

> **Colab Setup:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')
# Should show something like [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

## 1. Download Dataset from Kaggle

You need your Kaggle API key (`kaggle.json`).  
Get it from: **kaggle.com → Account → API → Create New Token**

In [ ]:
# Upload your kaggle.json when prompted
from google.colab import files
files.upload()  # upload kaggle.json

In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.system('cp kaggle.json /root/.kaggle/')
os.system('chmod 600 /root/.kaggle/kaggle.json')
os.system('pip install kaggle -q')
os.system('kaggle datasets download -d msambare/fer2013 --unzip -p /content/data')
print('Dataset ready.')

In [ ]:
# Mount Google Drive to save the trained model
from google.colab import drive
drive.mount('/content/drive')
SAVE_PATH = '/content/drive/MyDrive/emotion-detection/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'Model will be saved to: {SAVE_PATH}')

## 2. Imports & Config

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix

TRAIN_DIR = '/content/data/train'
TEST_DIR  = '/content/data/test'
EMOTIONS  = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
IMG_SIZE  = 224
BATCH     = 32

print('Config ready.')

## 3. Exploratory Data Analysis

In [ ]:
# Class distribution
train_counts = {e: len(os.listdir(os.path.join(TRAIN_DIR, e))) for e in EMOTIONS}
print('Training images per class:')
for e, c in train_counts.items():
    print(f'  {e:10s}: {c:,}')
print(f'\nTotal: {sum(train_counts.values()):,}')

plt.figure(figsize=(10, 4))
plt.bar(train_counts.keys(), train_counts.values(), color='steelblue')
plt.title('Training Images per Emotion')
plt.tight_layout()
plt.show()

In [ ]:
# Sample image per class
fig, axes = plt.subplots(1, 7, figsize=(16, 3))
for i, emotion in enumerate(EMOTIONS):
    img_path = os.path.join(TRAIN_DIR, emotion, os.listdir(os.path.join(TRAIN_DIR, emotion))[0])
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(emotion.capitalize())
    axes[i].axis('off')
plt.suptitle('Sample Image per Emotion')
plt.tight_layout()
plt.show()

## 4. Model 1 — Custom CNN from Scratch

In [ ]:
datagen = ImageDataGenerator(rescale=1./255)
train_gen = datagen.flow_from_directory(TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale', batch_size=BATCH, class_mode='categorical', shuffle=True)
test_gen = datagen.flow_from_directory(TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale', batch_size=BATCH, class_mode='categorical', shuffle=False)

def build_cnn(input_shape=(224, 224, 1)):
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(), layers.MaxPooling2D(2,2), layers.Dropout(0.25),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(), layers.MaxPooling2D(2,2), layers.Dropout(0.25),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(), layers.MaxPooling2D(2,2), layers.Dropout(0.25),
        layers.Flatten(),
        layers.Dense(256, activation='relu'), layers.Dropout(0.5),
        layers.Dense(7, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)
]

cnn_model = build_cnn()
history_cnn = cnn_model.fit(train_gen, epochs=30, validation_data=test_gen, callbacks=callbacks)

## 5. Model 2 — CNN with Image Augmentation

In [ ]:
aug_datagen = ImageDataGenerator(rescale=1./255, horizontal_flip=True,
    rotation_range=15, zoom_range=0.1, width_shift_range=0.1, height_shift_range=0.1)
aug_train_gen = aug_datagen.flow_from_directory(TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale', batch_size=BATCH, class_mode='categorical', shuffle=True)

cnn_aug = build_cnn()
history_aug = cnn_aug.fit(aug_train_gen, epochs=30, validation_data=test_gen, callbacks=callbacks)

## 6. Model 3 — Transfer Learning (ResNet50)

In [ ]:
rgb_datagen = ImageDataGenerator(rescale=1./255, horizontal_flip=True, rotation_range=15, zoom_range=0.1)
rgb_train = rgb_datagen.flow_from_directory(TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb', batch_size=BATCH, class_mode='categorical', shuffle=True)
rgb_test = ImageDataGenerator(rescale=1./255).flow_from_directory(TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb', batch_size=BATCH, class_mode='categorical', shuffle=False)

base = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

resnet = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])
resnet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

callbacks_resnet = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, verbose=1),
    ModelCheckpoint(SAVE_PATH + 'Final_Resnet50_Best_model.keras', save_best_only=True, verbose=1)
]

history_resnet = resnet.fit(rgb_train, epochs=30, validation_data=rgb_test, callbacks=callbacks_resnet)

## 7. Results & Comparison

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_title(f'{title} — Accuracy'); axes[0].legend()
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_title(f'{title} — Loss'); axes[1].legend()
    plt.tight_layout(); plt.show()

plot_history(history_cnn, 'Custom CNN')
plot_history(history_aug, 'CNN + Augmentation')
plot_history(history_resnet, 'ResNet50')

results = {
    'Custom CNN':          max(history_cnn.history['val_accuracy']),
    'CNN + Augmentation':  max(history_aug.history['val_accuracy']),
    'ResNet50 (Transfer)': max(history_resnet.history['val_accuracy'])
}
print('\nModel Comparison:')
for m, acc in results.items():
    print(f'  {m:25s}: {acc:.2%}')

plt.figure(figsize=(8,4))
plt.bar(results.keys(), results.values(), color=['#4C72B0','#55A868','#C44E52'])
plt.title('Validation Accuracy Comparison'); plt.ylabel('Accuracy'); plt.ylim(0,1)
plt.tight_layout(); plt.show()

In [ ]:
best = tf.keras.models.load_model(SAVE_PATH + 'Final_Resnet50_Best_model.keras')
y_pred = np.argmax(best.predict(rgb_test), axis=1)
y_true = rgb_test.classes

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9,7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[e.capitalize() for e in EMOTIONS],
            yticklabels=[e.capitalize() for e in EMOTIONS])
plt.title('Confusion Matrix — ResNet50')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

print(classification_report(y_true, y_pred, target_names=[e.capitalize() for e in EMOTIONS]))

In [ ]:
# Download model to your local machine
from google.colab import files
files.download(SAVE_PATH + 'Final_Resnet50_Best_model.keras')
print('Model downloaded. Replace the file in your emotion-detection project folder.')

## 8. Conclusions

- **ResNet50 transfer learning** outperformed both custom CNN variants
- **Image augmentation** reduced overfitting compared to the baseline CNN
- **Class imbalance** (disgust has fewest samples) affects per-class F1 — future work: class weights or oversampling
- **Deployment:** Best model served via Streamlit app with webcam + image upload